## Tutorial: Optimizing a Prompt

![TextGrad](https://github.com/vinid/data/blob/master/logo_full.png?raw=true)

An autograd engine -- for textual gradients!

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zou-group/TextGrad/blob/main/examples/notebooks/Prompt-Optimization.ipynb)
[![GitHub license](https://img.shields.io/badge/License-MIT-blue.svg)](https://lbesson.mit-license.org/)
[![Arxiv](https://img.shields.io/badge/arXiv-2406.07496-B31B1B.svg)](https://arxiv.org/abs/2406.07496)
[![Documentation Status](https://readthedocs.org/projects/textgrad/badge/?version=latest)](https://textgrad.readthedocs.io/en/latest/?badge=latest)
[![PyPI - Python Version](https://img.shields.io/pypi/pyversions/textgrad)](https://pypi.org/project/textgrad/)
[![PyPI](https://img.shields.io/pypi/v/textgrad)](https://pypi.org/project/textgrad/)

**Objectives:**

* In this tutorial, we will run prompt optimization.

**Requirements:**

* You need to have an OpenAI API key to run this tutorial. This should be set as an environment variable as OPENAI_API_KEY.


In [2]:
!pip install textgrad # you might need to restart the notebook after installing textgrad
import os
from google.colab import userdata

# 1. 设置 API Key（确保你已经在 Secrets 里存了 OPENAI_API_KEY）
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# 2. 设置中转站地址
# 注意：通常需要以 /v1 结尾，例如 https://api.yourproxy.com/v1
os.environ["OPENAI_BASE_URL"] = "https://api2.aigcbest.top/v1"
import argparse
import concurrent
from dotenv import load_dotenv
from tqdm import tqdm
import textgrad as tg
from textgrad.tasks import load_task
import numpy as np
import random
load_dotenv(override=True)


False

Let's first define some support functions

In [3]:
def set_seed(seed):
    np.random.seed(seed)
    random.seed(seed)

In [4]:
def eval_sample(item, eval_fn, model):
    """
    This function allows us to evaluate if an answer to a question in the prompt is a good answer.

    """
    x, y = item
    x = tg.Variable(x, requires_grad=False, role_description="query to the language model")
    y = tg.Variable(y, requires_grad=False, role_description="correct answer for the query")
    response = model(x)
    try:
        eval_output_variable = eval_fn(inputs=dict(prediction=response, ground_truth_answer=y))
        return int(eval_output_variable.value)
    except:
        eval_output_variable = eval_fn([x, y, response])
        eval_output_parsed = eval_fn.parse_output(eval_output_variable)
        return int(eval_output_parsed)

In [5]:
def eval_dataset(test_set, eval_fn, model, max_samples: int=None):
    if max_samples is None:
        max_samples = len(test_set)
    accuracy_list = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
        futures = []
        for _, sample in enumerate(test_set):

            future = executor.submit(eval_sample, sample, eval_fn, model)
            futures.append(future)
            if len(futures) >= max_samples:
                break
        tqdm_loader = tqdm(concurrent.futures.as_completed(futures), total=len(futures), position=0)
        for future in tqdm_loader:
            acc_item = future.result()
            accuracy_list.append(acc_item)
            tqdm_loader.set_description(f"Accuracy: {np.mean(accuracy_list)}")
    return accuracy_list

In [6]:
def run_validation_revert(system_prompt: tg.Variable, results, model, eval_fn, val_set):
    val_performance = np.mean(eval_dataset(val_set, eval_fn, model))
    previous_performance = np.mean(results["validation_acc"][-1])
    print("val_performance: ", val_performance)
    print("previous_performance: ", previous_performance)
    previous_prompt = results["prompt"][-1]

    if val_performance < previous_performance:
        print(f"rejected prompt: {system_prompt.value}")
        system_prompt.set_value(previous_prompt)
        val_performance = previous_performance

    results["validation_acc"].append(val_performance)

In [7]:
set_seed(12)
llm_api_eval = tg.get_engine(engine_name="gpt-4o")
llm_api_test = tg.get_engine(engine_name="gpt-3.5-turbo-0125")
tg.set_backward_engine(llm_api_eval, override=True)

# Load the data and the evaluation function
train_set, val_set, test_set, eval_fn = load_task("BBH_object_counting", evaluation_api=llm_api_eval)
print("Train/Val/Test Set Lengths: ", len(train_set), len(val_set), len(test_set))
STARTING_SYSTEM_PROMPT = train_set.get_task_description()


Train/Val/Test Set Lengths:  50 100 100


This is the system prompt we are going to start from:

In [8]:
print(STARTING_SYSTEM_PROMPT)


You will answer a reasoning question. Think step by step. The last line of your response should be of the following format: 'Answer: $VALUE' where VALUE is a numerical value.


In [10]:
train_loader = tg.tasks.DataLoader(train_set, batch_size=3, shuffle=True)


# Testing the 0-shot performance of the evaluation engine
system_prompt = tg.Variable(STARTING_SYSTEM_PROMPT,
                            requires_grad=True,
                            role_description="system prompt to the language model")
model_evaluation = tg.BlackboxLLM(llm_api_eval, system_prompt)

system_prompt = tg.Variable(STARTING_SYSTEM_PROMPT,
                            requires_grad=True,
                            role_description="structured system prompt to a somewhat capable language model that specifies the behavior and strategies for the QA task")
model = tg.BlackboxLLM(llm_api_test, system_prompt)

optimizer = tg.TextualGradientDescent(engine=llm_api_eval, parameters=[system_prompt])

results = {"test_acc": [], "prompt": [], "validation_acc": []}

# Convert numpy.int64 answers to standard Python int for all datasets
def convert_dataset_answers_to_int(dataset):
    converted_dataset = []
    for x, y in dataset:
        converted_dataset.append((x, int(y)))
    return converted_dataset

train_set = convert_dataset_answers_to_int(train_set)
val_set = convert_dataset_answers_to_int(val_set)
test_set = convert_dataset_answers_to_int(test_set)

results["test_acc"].append(eval_dataset(test_set, eval_fn, model))
results["validation_acc"].append(eval_dataset(val_set, eval_fn, model))
results["prompt"].append(system_prompt.get_value())

  0%|          | 0/100 [00:00<?, ?it/s]INFO:textgrad:LLMCall function forward
INFO:textgrad:StringBasedFunction
Accuracy: 1.0:   1%|          | 1/100 [00:03<05:30,  3.34s/it]INFO:textgrad:LLMCall function forward
INFO:textgrad:StringBasedFunction
Accuracy: 1.0:   1%|          | 1/100 [00:03<05:30,  3.34s/it]INFO:textgrad:LLMCall function forward
INFO:textgrad:StringBasedFunction
Accuracy: 1.0:   3%|▎         | 3/100 [00:05<02:28,  1.53s/it]INFO:textgrad:LLMCall function forward
INFO:textgrad:StringBasedFunction
Accuracy: 1.0:   4%|▍         | 4/100 [00:05<01:52,  1.18s/it]INFO:textgrad:LLMCall function forward
INFO:textgrad:StringBasedFunction
Accuracy: 1.0:   5%|▌         | 5/100 [00:06<01:56,  1.23s/it]INFO:textgrad:LLMCall function forward
INFO:textgrad:StringBasedFunction
Accuracy: 1.0:   6%|▌         | 6/100 [00:07<01:31,  1.03it/s]INFO:textgrad:LLMCall function forward
INFO:textgrad:StringBasedFunction
Accuracy: 1.0:   7%|▋         | 7/100 [00:08<01:38,  1.06s/it]INFO:textgrad:LL

In [12]:
for epoch in range(3):
    for steps, (batch_x, batch_y) in enumerate((pbar := tqdm(train_loader, position=0))):
        pbar.set_description(f"Training step {steps}. Epoch {epoch}")
        optimizer.zero_grad()
        losses = []
        for (x, y) in zip(batch_x, batch_y):
            x = tg.Variable(x, requires_grad=False, role_description="query to the language model")
            # Convert y to standard Python int before passing to tg.Variable
            y = tg.Variable(int(y), requires_grad=False, role_description="correct answer for the query")
            response = model(x)
            try:
                eval_output_variable = eval_fn(inputs=dict(prediction=response, ground_truth_answer=y))
            except:
                eval_output_variable = eval_fn([x, y, response])
            losses.append(eval_output_variable)
        total_loss = tg.sum(losses)
        total_loss.backward()
        optimizer.step()

        run_validation_revert(system_prompt, results, model, eval_fn, val_set)

        print("sys prompt: ", system_prompt)
        test_acc = eval_dataset(test_set, eval_fn, model)
        results["test_acc"].append(test_acc)
        results["prompt"].append(system_prompt.get_value())
        if steps == 3:
            break

Training step 0. Epoch 0: : 0it [00:00, ?it/s]INFO:textgrad:LLMCall function forward
INFO:textgrad:StringBasedFunction
INFO:textgrad:LLMCall function forward
INFO:textgrad:StringBasedFunction
INFO:textgrad:LLMCall function forward
INFO:textgrad:StringBasedFunction
INFO:textgrad:Idempotent backward
INFO:textgrad:Idempotent backward
INFO:textgrad:Idempotent backward
INFO:textgrad:_backward_through_string_fn prompt
INFO:textgrad:_backward_through_string_fn gradient
INFO:textgrad:_backward_through_llm prompt
INFO:textgrad:_backward_through_llm gradient
INFO:textgrad:_backward_through_string_fn prompt
INFO:textgrad:_backward_through_string_fn gradient
INFO:textgrad:_backward_through_llm prompt
INFO:textgrad:_backward_through_llm gradient
INFO:textgrad:_backward_through_string_fn prompt
INFO:textgrad:_backward_through_string_fn gradient
INFO:textgrad:_backward_through_llm prompt
INFO:textgrad:_backward_through_llm gradient
INFO:textgrad:TextualGradientDescent prompt for update
INFO:textgrad:

val_performance:  0.96
previous_performance:  0.98
rejected prompt: You will answer a reasoning question. Provide a concise answer by directly calculating the total count of items that belong to the specified category. Use the format 'Total count: $VALUE' for your answer, where VALUE is a numerical value. Ensure the response format is consistent and error-free. If there is uncertainty about the classification of an item, ask for clarification from the user. Implement a check for duplicates or non-relevant items and adjust the count accordingly.
sys prompt:  You will answer a reasoning question. Think step by step. The last line of your response should be of the following format: 'Answer: $VALUE' where VALUE is a numerical value.


Accuracy: 1.0:   0%|          | 0/100 [00:00<?, ?it/s]INFO:textgrad:StringBasedFunction
INFO:textgrad:LLMCall function forward
Accuracy: 1.0:   0%|          | 0/100 [00:00<?, ?it/s]INFO:textgrad:StringBasedFunction
INFO:textgrad:LLMCall function forward
Accuracy: 1.0:   0%|          | 0/100 [00:00<?, ?it/s]INFO:textgrad:StringBasedFunction
INFO:textgrad:LLMCall function forward
Accuracy: 1.0:   0%|          | 0/100 [00:00<?, ?it/s]INFO:textgrad:StringBasedFunction
INFO:textgrad:LLMCall function forward
Accuracy: 1.0:   0%|          | 0/100 [00:00<?, ?it/s]INFO:textgrad:StringBasedFunction
INFO:textgrad:LLMCall function forward
Accuracy: 1.0:   0%|          | 0/100 [00:00<?, ?it/s]INFO:textgrad:LLMCall function forward
INFO:textgrad:StringBasedFunction
INFO:textgrad:LLMCall function forward
Accuracy: 1.0:   0%|          | 0/100 [00:00<?, ?it/s]INFO:textgrad:StringBasedFunction
INFO:textgrad:LLMCall function forward
Accuracy: 1.0:   0%|          | 0/100 [00:00<?, ?it/s]INFO:textgrad:Stri

val_performance:  0.99
previous_performance:  0.98
sys prompt:  You will answer a reasoning question. Follow these steps to ensure clarity and accuracy:

1. **Explicit Enumeration**: List each item and its count explicitly, using a structured format such as bullet points or a table. Ensure each item is numbered sequentially without gaps.

2. **Complete List Verification**: Verify that all items are listed and numbered correctly before concluding.

3. **Step-by-Step Calculation**: Present each calculation step in a separate line or table format. List each item and its count separately, then show the addition process in a step-by-step manner.

4. **Simplification Directive**: Provide the total count directly after listing the items, unless intermediate steps are necessary for clarity. Focus on numerical values and calculations, minimizing descriptive text.

5. **Error Detection and Reporting**: If any items are missing or numbering is inconsistent, report the issue and suggest correction

  0%|          | 0/100 [00:00<?, ?it/s]INFO:textgrad:LLMCall function forward
INFO:textgrad:StringBasedFunction
Accuracy: 1.0:   1%|          | 1/100 [00:01<02:59,  1.82s/it]INFO:textgrad:LLMCall function forward
INFO:textgrad:StringBasedFunction
Accuracy: 1.0:   1%|          | 1/100 [00:01<02:59,  1.82s/it]INFO:textgrad:LLMCall function forward
INFO:textgrad:StringBasedFunction
Accuracy: 1.0:   3%|▎         | 3/100 [00:02<01:21,  1.18it/s]INFO:textgrad:LLMCall function forward
INFO:textgrad:StringBasedFunction
Accuracy: 1.0:   4%|▍         | 4/100 [00:03<01:25,  1.13it/s]INFO:textgrad:LLMCall function forward
INFO:textgrad:StringBasedFunction
Accuracy: 1.0:   5%|▌         | 5/100 [00:04<01:10,  1.34it/s]INFO:textgrad:LLMCall function forward
INFO:textgrad:StringBasedFunction
Accuracy: 1.0:   6%|▌         | 6/100 [00:05<01:26,  1.09it/s]INFO:textgrad:LLMCall function forward
INFO:textgrad:StringBasedFunction
Accuracy: 1.0:   7%|▋         | 7/100 [00:05<01:11,  1.29it/s]INFO:textgrad:LL

...

val_performance:  0.99
previous_performance:  1.0
rejected prompt: You will answer a reasoning question. Follow these steps to ensure clarity, accuracy, and transparency:

1. **Explicit Enumeration**: List each item and its count explicitly, using a structured format such as bullet points or a table. Use the format 'Item: Count' consistently, with a space after the colon for enhanced readability. Ensure each item is numbered sequentially without gaps.

2. **Complete List Verification**: Verify that all items are listed and numbered correctly before concluding. Ensure no items are missing and all counts are accurate. Handle potential errors in the input data by checking for non-numeric values or missing quantities, and provide an appropriate error message or correction.

3. **Explicit Calculation Steps**: Include detailed calculation steps in your response. Break down the summation process explicitly, listing each item and its count in the calculation line. For example, "Total items = 1

INFO:textgrad:LLMCall function forward
INFO:textgrad:LLMCall function forward
  0%|          | 0/100 [00:00<?, ?it/s]INFO:textgrad:StringBasedFunction
INFO:textgrad:StringBasedFunction
Accuracy: 1.0:   0%|          | 0/100 [00:00<?, ?it/s]INFO:textgrad:LLMCall function forward
INFO:textgrad:LLMCall function forward
INFO:textgrad:StringBasedFunction
INFO:textgrad:StringBasedFunction
Accuracy: 1.0:   0%|          | 0/100 [00:00<?, ?it/s]INFO:textgrad:LLMCall function forward
INFO:textgrad:LLMCall function forward
INFO:textgrad:StringBasedFunction
Accuracy: 1.0:   0%|          | 0/100 [00:00<?, ?it/s]INFO:textgrad:StringBasedFunction
INFO:textgrad:LLMCall function forward
Accuracy: 1.0:   0%|          | 0/100 [00:00<?, ?it/s]INFO:textgrad:StringBasedFunction
INFO:textgrad:LLMCall function forward
Accuracy: 1.0:   0%|          | 0/100 [00:00<?, ?it/s]INFO:textgrad:StringBasedFunction
INFO:textgrad:LLMCall function forward
Accuracy: 1.0:   0%|          | 0/100 [00:00<?, ?it/s]INFO:textgrad:

val_performance:  0.95
previous_performance:  1.0
rejected prompt: You will answer a reasoning question. Follow these steps to ensure clarity, accuracy, and transparency:

1. **Explicit Enumeration**: List each item and its count explicitly using a structured format such as bullet points. Use the format 'Item: Count' consistently, with a space after the colon for enhanced readability. Ensure each item is numbered sequentially without gaps.

2. **Consistent Delimiters**: Use a consistent delimiter, such as a comma or semicolon, to separate items and counts. This facilitates easier parsing and verification.

3. **Simplified Calculation Steps**: Present mathematical expressions in a minimalistic manner. List items and their counts clearly, allowing the function to handle the summation. Avoid detailed summation steps unless necessary for clarity.

4. **Streamlined Output Format**: Ensure the final line of your response is formatted as 'Answer: VALUE' with no additional text. Provide a clea

INFO:textgrad:StringBasedFunction
Accuracy: 1.0:   0%|          | 0/100 [00:00<?, ?it/s]INFO:textgrad:LLMCall function forward
INFO:textgrad:StringBasedFunction
INFO:textgrad:LLMCall function forward
Accuracy: 1.0:   0%|          | 0/100 [00:00<?, ?it/s]INFO:textgrad:LLMCall function forward
INFO:textgrad:StringBasedFunction
Accuracy: 1.0:   0%|          | 0/100 [00:00<?, ?it/s]INFO:textgrad:LLMCall function forward
INFO:textgrad:StringBasedFunction
INFO:textgrad:LLMCall function forward
Accuracy: 1.0:   0%|          | 0/100 [00:00<?, ?it/s]INFO:textgrad:StringBasedFunction
INFO:textgrad:LLMCall function forward
Accuracy: 1.0:   0%|          | 0/100 [00:00<?, ?it/s]INFO:textgrad:StringBasedFunction
INFO:textgrad:LLMCall function forward
Accuracy: 1.0:   0%|          | 0/100 [00:00<?, ?it/s]INFO:textgrad:LLMCall function forward
INFO:textgrad:StringBasedFunction
INFO:textgrad:LLMCall function forward
Accuracy: 1.0:   0%|          | 0/100 [00:00<?, ?it/s]INFO:textgrad:StringBasedFunction

KeyboardInterrupt: 